### RAG Pipeline Data Ingestion to Vector DB pipeline - krish_naik_YT_ref

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
### Read all the pdf inside the directory 
def process_all_pdfs(pdf_directory):
    """Process all pdf files in a directory"""
    all_documents = [] #to store all documents as list
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"found {len(pdf_files)} pdf file to process")

    for pdf_file in pdf_files:
        print(f"processsing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
            
            all_documents.extend(documents)
            print(f" ✅ Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f" ❌ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")


In [ ]:
all_pdf_documents

In [8]:
### Text splitting getting into chunks
def split_documents(documents, chunk_size=1000,chunk_overlap=200):
    """split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

     # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [ ]:
chunks=split_documents(all_pdf_documents)
chunks

### Embedding & Vector Store DB

### Embedding library - Sentence-Transformers - model from huggingface is used
### VectorStore - Faiss-cpu & open source chromadb

In [10]:
# Importing Libraries
import numpy as np
from sentence_transformers import SentenceTransformer #Embedding model will be available inside this
import chromadb #Vector Database to store the embeddings
from chromadb.config import Settings
import uuid #Every record insierted into vector db has some kind of unique id, it's generated using uuid library
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity #Used for cosine similarity or similarity search in the vector db 

In [ ]:
class EmbeddingManager:
    """This class handles document embedding generation using SentencceTransformer"""
    """Every class has it's init function"""
    def __init__(self, model_name: str = "all-MiniLM-L6-V2"): 
        """ Intialize the embedding manager
        Args:
            model_name: Hugging face model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model() #this is a function to load the model, below the function is created and defined

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            #print(f"Model loaded Successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}") #Every text in the senntence will be converted embedding in a 384 dimension
        except Exception as e:
            print(f"Error loading model: {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray: #this converts the list of string into array of numbers/embeddings
        """Generate Embeddings for a list of texts
        
        Args:
            texts: List of text strings to embedding
        
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embedding for {len(texts)}texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generate Embeddings with shape: {embeddings.shape}")
        return embeddings
        
## Initialize the Embessing Manager

embedding_manager = EmbeddingManager()
embedding_manager



### Vector Store

In [ ]:
class VectorStore:
    """Manages document embeddings in a chroma db vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory = "../data/vector_store"):
        """
        Initialize the Vector Store

        Args:
            collection_name: Name of the chromaDB collection
            persistent_directory: Directory to persist the chroma DB data
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize chromaDB client and collection"""
        try:
            #Create persistent chromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"decription": "PDF document embedding for RAG"} 
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
           Add documents and embeddings into the vector store

           Args:
               documents: List of langchain documents
               embeddings: Corresponding embedding of the documents 
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to the vector store")
    
        #Prepare data for chromaDB
        ids = []
        metadatas = []
        documents_txt = []
        embeddings_list = []

        for i, (doc, embeddings) in enumerate(zip(documents,embeddings)):
            #Generate unique ID 
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare Metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #Document Content
            documents_txt.append(doc.page_content)

            #Embeddings
            embeddings_list.append(embeddings.tolist())

        #Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_txt
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    


In [ ]:
chunks

In [ ]:
###Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the embeddings
embeddings=embedding_manager.generate_embeddings(texts)

## store in the vecctor database
vectorstore.add_documents(chunks,embeddings)

In [24]:
class RAGRetriever:
    """Handles query based retrieval from the vector store - The retriever is a simple interface based on what query is inputed it retrieve info from vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the RAG retriever
        
        Args:
            vetor_store: vectore store containing the document embeddings
            embedding_manager: manager for generating the query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str,Any]]: #This is the defined retrieve function which performa extracting info from vector stor based on query
        """
        Retrieve relevant documents for a query

        Args:
            query: the search query
            top_k: number of top matching results to return
            score_threshold: minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata

        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top k: {top_k}, Score threshold: {score_threshold}")

        #Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0] #The given query is converted into embedding

        #Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()], #the embedded query is passed as a list to the vector_store
                n_results=top_k
            )

            #Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id,document,metadata,distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    #Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                
                print(f"retrieved {len(retrieved_docs)} documents after applying score threshold")
            else:
                print("No document found")

            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGRetriever(vectorstore,embedding_manager)

In [ ]:
rag_retriever

In [ ]:
rag_retriever.retrieve("Why Mercedez-Benz relocated to germany")